# 01 Data Preparation
### Prepare data for training
---

### Importing data

##### Importing essential libraries

In [3]:
import pandas as pd
import yfinance as yf

##### Importing data and Creating the DataFrame

In [4]:
#Downloading data from yfinance and storing in df as a DataFrame
df = pd.DataFrame(yf.download("VOLV-B.ST", start="2020-01-01", end="2024-12-31"))
#Checking if it downloaded and stored correctly
df.head()

C:\Users\Gymnasiet\AppData\Local\Temp\ipykernel_1436\2020319533.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = pd.DataFrame(yf.download("VOLV-B.ST", start="2020-01-01", end="2024-12-31"))
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,VOLV-B.ST,VOLV-B.ST,VOLV-B.ST,VOLV-B.ST,VOLV-B.ST
Date,,,,,
2020-01-02,109.299957,109.435943,107.022162,107.022162,3251859
2020-01-03,106.886192,108.450051,106.240254,108.450051,4346445
2020-01-07,107.804115,108.620038,106.716211,106.784209,4889908
2020-01-08,108.450050,108.552037,105.968282,106.886192,4038628
2020-01-09,106.614220,109.707938,106.342238,109.469956,6003067


### Cleaning data

##### Removing unecessary data

In [5]:
#Removing all columns except 'Close'
df.drop(['High','Low','Open','Volume'], inplace=True, axis=1)
#Check
df.head()


Price,Close
Ticker,VOLV-B.ST
Date,
2020-01-02,109.299957
2020-01-03,106.886192
2020-01-07,107.804115
2020-01-08,108.450050
2020-01-09,106.614220


##### Removing Ticker

In [6]:
#Turns multiindex into single index i.e. removes ticker 
df.columns = df.columns.get_level_values(0)
#Check
df.head()

Price,Close
Date,
2020-01-02,109.299957
2020-01-03,106.886192
2020-01-07,107.804115
2020-01-08,108.450050
2020-01-09,106.614220


### Adding training data

##### 5-day lagged return

In [7]:
#Creating new column with percentage change in Close form previous day
df["Return_Lag_1"] = df["Close"].pct_change()

#Creating 4 new columns 2-5 with value taken from Return_Lag_1 column with a shift 
for i in range(2,6):
    df[f"Return_Lag_{i}"] = df["Return_Lag_1"].shift(i-1)
#Check
df.head(6)

Price,Close,Return_Lag_1,Return_Lag_2,Return_Lag_3,Return_Lag_4,Return_Lag_5
Date,,,,,,
2020-01-02,109.299957,NaN,NaN,NaN,NaN,NaN
2020-01-03,106.886192,-0.022084,NaN,NaN,NaN,NaN
2020-01-07,107.804115,0.008588,-0.022084,NaN,NaN,NaN
2020-01-08,108.450050,0.005992,0.008588,-0.022084,NaN,NaN
2020-01-09,106.614220,-0.016928,0.005992,0.008588,-0.022084,NaN
2020-01-10,106.104263,-0.004783,-0.016928,0.005992,0.008588,-0.022084


##### 5-day moving average

In [8]:
#Taking mean with .mean() of the five latest days with .rolling(5) and adding it to column MA5
df[("MA5")] = df[("Close")].rolling(5).mean()
#Check
df.head(6)


Price,Close,Return_Lag_1,Return_Lag_2,Return_Lag_3,Return_Lag_4,Return_Lag_5,MA5
Date,,,,,,,
2020-01-02,109.299957,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,106.886192,-0.022084,NaN,NaN,NaN,NaN,NaN
2020-01-07,107.804115,0.008588,-0.022084,NaN,NaN,NaN,NaN
2020-01-08,108.450050,0.005992,0.008588,-0.022084,NaN,NaN,NaN
2020-01-09,106.614220,-0.016928,0.005992,0.008588,-0.022084,NaN,107.810907
2020-01-10,106.104263,-0.004783,-0.016928,0.005992,0.008588,-0.022084,107.171768


##### Removing Nan values

In [9]:
#Dropping rows with Nan values
df.dropna(inplace=True)
#Check
df.head()

Price,Close,Return_Lag_1,Return_Lag_2,Return_Lag_3,Return_Lag_4,Return_Lag_5,MA5
Date,,,,,,,
2020-01-10,106.104263,-0.004783,-0.016928,0.005992,0.008588,-0.022084,107.171768
2020-01-13,105.934288,-0.001602,-0.004783,-0.016928,0.005992,0.008588,106.981387
2020-01-14,106.410240,0.004493,-0.001602,-0.004783,-0.016928,0.005992,106.702612
2020-01-15,106.002266,-0.003834,0.004493,-0.001602,-0.004783,-0.016928,106.213055
2020-01-16,106.852196,0.008018,-0.003834,0.004493,-0.001602,-0.004783,106.260651


### Saving data

##### Preparing data for saving

In [10]:
#Turning date into column instead of index to remove nan values when csv filed is used later
df.reset_index()

Price,Date,Close,Return_Lag_1,Return_Lag_2,Return_Lag_3,Return_Lag_4,Return_Lag_5,MA5
0,2020-01-10,106.104263,-0.004783,-0.016928,0.005992,0.008588,-0.022084,107.171768
1,2020-01-13,105.934288,-0.001602,-0.004783,-0.016928,0.005992,0.008588,106.981387
2,2020-01-14,106.410240,0.004493,-0.001602,-0.004783,-0.016928,0.005992,106.702612
3,2020-01-15,106.002266,-0.003834,0.004493,-0.001602,-0.004783,-0.016928,106.213055
4,2020-01-16,106.852196,0.008018,-0.003834,0.004493,-0.001602,-0.004783,106.260651
...,...,...,...,...,...,...,...,...
1250,2024-12-19,252.806015,-0.020682,-0.000363,0.002910,-0.019615,-0.009187,257.864023
1251,2024-12-20,250.745377,-0.008151,-0.020682,-0.000363,0.002910,-0.019615,255.484894
1252,2024-12-23,249.340378,-0.005603,-0.008151,-0.020682,-0.000363,0.002910,253.855099
1253,2024-12-27,251.401031,0.008264,-0.005603,-0.008151,-0.020682,-0.000363,252.487564


##### Saving data in a .csv file at /data/volvo_2020_2024_clean.csv

In [ ]:
#Storing df in data folder
df.to_csv("../data/volvo_2020_2024_clean.csv")